# WildTrack cross-camera MTMC comparison

This notebook compares camera-local tracker identities with shared MTMC identities. It reads the persisted offline evaluation artifact, reports image-plane and ground-plane metrics, inspects held-out homography error by image row, and visualises the association-parameter sweep. Missing optional artifacts are reported in-place instead of failing the rest of the notebook.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)


def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / "metrics").is_dir() and (path / "configs").is_dir():
            return path
    return candidate


REPO_ROOT = find_repo_root()
RESULT_PATH = REPO_ROOT / "metrics/results/mtmc_offline.json"
HOMOGRAPHY_FIT_PATH = REPO_ROOT / "metrics/results/homography_fit.json"
HOMOGRAPHY_CONFIG_DIR = REPO_ROOT / "configs"
WILDTRACK_ROOT: Path | None = None  # Override if the path recorded by the fit moved.
SWEEP_METRIC = "xcam_f1"  # Also try pooled_IDF1, ground_MODA, or ground_MODP.
IMAGE_ROW_BINS = 10

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")
print(f"MTMC results: {RESULT_PATH}")

In [ ]:
def load_json_artifact(path: Path, label: str) -> dict:
    if not path.exists():
        print(f"{label} is not available at {path}. Generate it, then rerun this cell.")
        return {}
    try:
        payload = json.loads(path.read_text())
    except (OSError, json.JSONDecodeError) as exc:
        print(f"Could not load {label} from {path}: {exc}")
        return {}
    print(f"Loaded {label} from {path}")
    return payload


report = load_json_artifact(RESULT_PATH, "MTMC evaluation")
metric_root = report.get("metrics", report) if report else {}
fit_report = load_json_artifact(HOMOGRAPHY_FIT_PATH, "homography fit summary")

## Image-plane identity quality

Per-camera IDF1 scores the tracker-native `object_id` independently in each view. Pooled IDF1 interleaves camera images and scores the shared `global_id`, so it exposes cross-camera identity mistakes. Pooled MOTA/MOTP remain detection-weighted camera aggregates and are intentionally not presented as cross-camera identity measures.

In [ ]:
image_plane = metric_root.get("image_plane", {})
per_camera = image_plane.get("per_camera", {})
pooled = image_plane.get("pooled", {})

idf1_rows = [
    {"scope": f"Camera {source_id}", "IDF1": values.get("IDF1", np.nan)}
    for source_id, values in sorted(per_camera.items(), key=lambda item: str(item[0]))
]
if "IDF1" in pooled:
    idf1_rows.append({"scope": "Pooled global_id", "IDF1": pooled["IDF1"]})

if not idf1_rows:
    print("No image-plane metrics found. Run metrics/evaluate_mtmc.py with --output-json.")
else:
    idf1_df = pd.DataFrame(idf1_rows)
    display(idf1_df.style.format({"IDF1": "{:.3f}"}))
    colors = ["#4c78a8"] * max(0, len(idf1_df) - 1) + ["#f58518"]
    ax = idf1_df.plot.bar(x="scope", y="IDF1", color=colors, legend=False, figsize=(8, 4))
    ax.set(title="Camera-local object_id vs pooled global_id", xlabel="", ylabel="IDF1", ylim=(0, 1.05))
    ax.tick_params(axis="x", rotation=0)
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
    plt.tight_layout()
    plt.show()

In [ ]:
cross_camera = metric_root.get("cross_camera", {})
xcam_columns = {
    "Precision": "xcam_precision",
    "Recall": "xcam_recall",
    "F1": "xcam_f1",
}
xcam_rows = [
    {"metric": label, "score": cross_camera[key]}
    for label, key in xcam_columns.items()
    if key in cross_camera
]

if not xcam_rows:
    print("No cross-camera identity metrics found in the evaluation artifact.")
else:
    xcam_df = pd.DataFrame(xcam_rows)
    display(xcam_df.style.format({"score": "{:.3f}"}))
    ax = xcam_df.plot.bar(x="metric", y="score", color=["#54a24b", "#e45756", "#72b7b2"], legend=False, figsize=(7, 4))
    ax.set(title="Cross-camera identity agreement", xlabel="", ylabel="Score", ylim=(0, 1.05))
    ax.tick_params(axis="x", rotation=0)
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
    plt.tight_layout()
    plt.show()
    counts = {key: cross_camera.get(key) for key in ("xcam_tp", "xcam_fp", "xcam_fn") if key in cross_camera}
    if counts:
        print("Pair counts:", counts)

## Ground-plane quality at both distance gates

Ground-plane predictions are fused with inverse-variance weights before scoring. The 0.5 m gate is the primary operating point; 1.0 m is the requested sensitivity check.

In [ ]:
ground_plane = metric_root.get("ground_plane", {})
ground_rows = [
    {
        "gate": gate,
        "gate_m": values.get("gate_m", float(str(gate).removesuffix("m"))),
        "MODA": values.get("MODA", np.nan),
        "MODP": values.get("MODP", np.nan),
        "precision": values.get("precision", np.nan),
        "recall": values.get("recall", np.nan),
    }
    for gate, values in ground_plane.items()
]
ground_rows.sort(key=lambda row: row["gate_m"])

if not ground_rows:
    print("No ground-plane metrics found. Evaluate with --ground-gates 0.5 1.0.")
else:
    ground_df = pd.DataFrame(ground_rows)
    display(ground_df.style.format({key: "{:.3f}" for key in ("gate_m", "MODA", "MODP", "precision", "recall")}))
    ax = ground_df.set_index("gate")[["MODA", "MODP"]].plot.bar(
        color=["#4c78a8", "#f58518"], figsize=(8, 4), rot=0
    )
    plotted_scores = ground_df[["MODA", "MODP"]].to_numpy(dtype=float)
    finite_scores = plotted_scores[np.isfinite(plotted_scores)]
    score_min = float(finite_scores.min()) if finite_scores.size else 0.0
    y_min = min(0.0, score_min * 1.15)
    ax.set(title="Ground-plane accuracy by matching gate", xlabel="Distance gate", ylabel="Score", ylim=(y_min, 1.05))
    ax.axhline(0.0, color="#666666", linewidth=0.8)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3)
    plt.tight_layout()
    plt.show()

## Held-out projection error versus image row

The curve uses held-out frames only. If the fit artifact contains row-level errors, those are read directly. The current aggregate artifact can also be expanded reproducibly from its recorded WildTrack root, held-out frame list, and each saved homography. Solid lines are median error within an image-row bin; the shaded band reaches the 90th percentile.

In [ ]:
def projection_fit_summary(payload: dict) -> pd.DataFrame:
    rows = []
    for camera, values in payload.get("cameras", {}).items():
        rows.append(
            {
                "camera": camera,
                "holdout_points": values.get("n_holdout_points"),
                "median_m": values.get("holdout_median_m"),
                "p90_m": values.get("holdout_p90_m"),
                "quality_gate_passed": values.get("quality_gate_passed"),
            }
        )
    return pd.DataFrame(rows)


fit_summary_df = projection_fit_summary(fit_report)
if fit_summary_df.empty:
    print("No homography fit summary is available.")
else:
    display(fit_summary_df.style.format({"median_m": "{:.4f}", "p90_m": "{:.4f}"}))

In [ ]:
def _precomputed_projection_rows(payload: dict) -> tuple[list[dict], set[str]]:
    rows: list[dict] = []
    covered: set[str] = set()
    for camera, values in payload.get("cameras", {}).items():
        samples = values.get("projection_error_by_image_row", [])
        for sample in samples if isinstance(samples, list) else []:
            image_row = sample.get("image_row_px", sample.get("image_row"))
            error_m = sample.get("error_m", sample.get("projection_error_m"))
            if image_row is None or error_m is None:
                continue
            rows.append(
                {
                    "camera": camera,
                    "image_row_px": float(image_row),
                    "image_height": float(sample.get("image_height", 1080)),
                    "error_m": float(error_m),
                }
            )
            covered.add(camera)
    return rows, covered


def load_projection_error_rows(payload: dict) -> tuple[pd.DataFrame, list[str]]:
    rows, covered = _precomputed_projection_rows(payload)
    messages: list[str] = []
    camera_summaries = payload.get("cameras", {})
    config_paths = sorted(HOMOGRAPHY_CONFIG_DIR.glob("homography_C*.json"))
    camera_names = sorted(set(camera_summaries) | {path.stem.removeprefix("homography_") for path in config_paths})

    recorded_root = payload.get("dataset_root")
    dataset_root = Path(WILDTRACK_ROOT).expanduser() if WILDTRACK_ROOT is not None else (Path(recorded_root) if recorded_root else None)
    if dataset_root is None or not dataset_root.exists():
        if camera_names and set(camera_names) - covered:
            messages.append(
                "Row-level samples are not embedded and the recorded WildTrack root is unavailable. "
                "Set WILDTRACK_ROOT in the configuration cell to reconstruct the curve."
            )
        return pd.DataFrame(rows), messages

    try:
        from metrics.wildtrack_gt import load_wildtrack_mtmc_gt
        from pipelines.ground_plane import foot_point, is_foot_reliable, project_point
    except ImportError as exc:
        messages.append(f"Could not import projection helpers: {exc}")
        return pd.DataFrame(rows), messages

    for camera in camera_names:
        if camera in covered:
            continue
        config_path = HOMOGRAPHY_CONFIG_DIR / f"homography_{camera}.json"
        if not config_path.exists():
            messages.append(f"Skipping {camera}: no saved homography at {config_path}.")
            continue
        try:
            config = json.loads(config_path.read_text())
            fit = camera_summaries.get(camera, config.get("fit", {}))
            holdout_frames = set(fit.get("holdout_frames", config.get("fit", {}).get("holdout_frames", [])))
            if not holdout_frames:
                messages.append(f"Skipping {camera}: the fit artifact has no held-out frame list.")
                continue
            source_id = int(config["source_id"])
            matrix = np.asarray(config["matrix"], dtype=float)
            camera_rows = load_wildtrack_mtmc_gt(dataset_root, [camera], {camera: source_id})
            for row in camera_rows:
                if int(row["frame_num"]) not in holdout_frames:
                    continue
                if not is_foot_reliable(
                    row["left"], row["top"], row["width"], row["height"],
                    image_width=int(row["image_width"]), image_height=int(row["image_height"]),
                ):
                    continue
                u, v = foot_point(row["left"], row["top"], row["width"], row["height"])
                projected = np.asarray(project_point(matrix, u, v))
                expected = np.asarray([row["world_x"], row["world_y"]])
                rows.append(
                    {
                        "camera": camera,
                        "image_row_px": v,
                        "image_height": int(row["image_height"]),
                        "error_m": float(np.linalg.norm(projected - expected)),
                    }
                )
        except (OSError, ValueError, KeyError, json.JSONDecodeError) as exc:
            messages.append(f"Skipping {camera}: {exc}")
    return pd.DataFrame(rows), messages


projection_df, projection_messages = load_projection_error_rows(fit_report)
for message in projection_messages:
    print(message)
print(f"Loaded {len(projection_df):,} held-out projection-error samples.") if not projection_df.empty else None

In [ ]:
if projection_df.empty:
    print("Projection error versus image row cannot be plotted until row-level samples or the WildTrack dataset are available.")
else:
    cameras = sorted(projection_df["camera"].unique())
    fig, axes = plt.subplots(1, len(cameras), figsize=(5 * len(cameras), 4), sharey=True, squeeze=False)
    for ax, camera in zip(axes[0], cameras):
        camera_df = projection_df[projection_df["camera"] == camera].copy()
        image_height = float(camera_df["image_height"].max())
        edges = np.linspace(0.0, image_height, IMAGE_ROW_BINS + 1)
        camera_df["row_bin"] = pd.cut(camera_df["image_row_px"], bins=edges, include_lowest=True)
        binned = camera_df.groupby("row_bin", observed=True)["error_m"].agg(
            median="median", p90=lambda values: values.quantile(0.9), samples="size"
        ).reset_index()
        binned["row_px"] = binned["row_bin"].map(lambda interval: interval.mid).astype(float)
        ax.plot(binned["row_px"], binned["median"], marker="o", color="#4c78a8", label="Median")
        ax.fill_between(binned["row_px"], binned["median"], binned["p90"], color="#4c78a8", alpha=0.2, label="Median–p90")
        ax.set(title=f"{camera} ({len(camera_df):,} points)", xlabel="Foot-point image row (px)", xlim=(0, image_height))
        ax.grid(alpha=0.2)
        ax.legend()
    axes[0][0].set_ylabel("Held-out projection error (m)")
    fig.suptitle("Homography error versus image row")
    fig.tight_layout()
    plt.show()

## `min_affinity × z_gate` sweep surface

The surface defaults to cross-camera F1. Change `SWEEP_METRIC` in the configuration cell to inspect pooled IDF1 or either ground-plane metric from the same persisted sweep.

In [ ]:
sweep_rows = report.get("sweep", []) if report else []
if not sweep_rows:
    print("No parameter sweep found. Re-run metrics/evaluate_mtmc.py with --sweep and --output-json.")
else:
    sweep_df = pd.DataFrame(sweep_rows)
    if SWEEP_METRIC not in sweep_df.columns:
        available = sorted(set(sweep_df.columns) - {"min_affinity", "z_gate"})
        print(f"Sweep metric {SWEEP_METRIC!r} is unavailable. Choose one of: {available}")
    else:
        surface = sweep_df.pivot(index="z_gate", columns="min_affinity", values=SWEEP_METRIC).sort_index().sort_index(axis=1)
        display(surface.style.format("{:.3f}").background_gradient(cmap="viridis", axis=None))
        x_values = surface.columns.to_numpy(dtype=float)
        y_values = surface.index.to_numpy(dtype=float)
        x_grid, y_grid = np.meshgrid(x_values, y_values)
        z_grid = surface.to_numpy(dtype=float)
        fig = plt.figure(figsize=(9, 6))
        ax = fig.add_subplot(111, projection="3d")
        if np.isfinite(z_grid).all() and min(z_grid.shape) >= 2:
            plotted = ax.plot_surface(x_grid, y_grid, z_grid, cmap="viridis", edgecolor="white", linewidth=0.5)
        else:
            valid = np.isfinite(z_grid)
            plotted = ax.scatter(x_grid[valid], y_grid[valid], z_grid[valid], c=z_grid[valid], cmap="viridis", s=80)
            print("The sweep grid is incomplete; showing available parameter combinations as points.")
        ax.set(xlabel="min_affinity", ylabel="z_gate", zlabel=SWEEP_METRIC, title=f"MTMC association sweep: {SWEEP_METRIC}")
        fig.colorbar(plotted, ax=ax, shrink=0.65, pad=0.12, label=SWEEP_METRIC)
        plt.tight_layout()
        plt.show()

## Reading the comparison

A healthy result keeps camera-local IDF1 stable while raising pooled IDF1 and cross-camera F1. Compare both ground-plane gates: a large gain at 1.0 m with weak 0.5 m MODA usually points to projection or detector foot-point error rather than association alone. The projection-row curves help localise that error, while a broad high-scoring region on the sweep surface is preferable to a single sharp optimum.